In [ ]:
import os
import re
import pandas as pd
import cv2
import numpy as np

PNG_ROOT = 'voxelmorph/data/image slice-T1'
TARGET_SIZE = (224, 192)
NUM_PAIRS = 2000

def parse_numeric(filename):
    match = re.match(r'^(\d+)\.png$', filename)
    return int(match.group(1)) if match else -1

def find_patient_directories(root):
    return sorted(
        os.path.join(root, d)
        for d in os.listdir(root)
        if os.path.isdir(os.path.join(root, d))
    )

def load_volumes(patient_dirs, target_size):
    vols = []
    for d in patient_dirs:
        files = sorted(
            (f for f in os.listdir(d) if f.lower().endswith('.png')),
            key=parse_numeric
        )
        slices = []
        for f in files:
            img = cv2.imread(os.path.join(d, f), cv2.IMREAD_GRAYSCALE)
            if img is None or img.max() == 0:
                continue
            resized = cv2.resize(img, target_size, interpolation=cv2.INTER_LINEAR)
            slices.append(resized.astype(np.float32) / 255.0)
        if slices:
            vols.append(np.stack(slices, axis=-1))
    return vols

def make_inter_subject_pairs(data, num_pairs):
    N = data.shape[0]
    m, f = [], []
    for _ in range(num_pairs):
        i, j = np.random.choice(N, 2, replace=False)
        m.append(data[i])
        f.append(data[j])
    return np.array(m), np.array(f)

patient_dirs = find_patient_directories(PNG_ROOT)
volumes = load_volumes(patient_dirs, TARGET_SIZE)
mid = volumes[0].shape[-1] // 2
data = np.stack([v[:, :, mid] for v in volumes if mid < v.shape[-1]], axis=0)[..., np.newaxis]

moving, fixed = make_inter_subject_pairs(data, NUM_PAIRS)
zeros = np.zeros_like(moving)

np.savez("ixi_pairs.npz", moving=moving, fixed=fixed, zeros=zeros)
print(f"Built {moving.shape[0]} inter-subject training pairs.")

In [ ]:
data = np.load("ixi_pairs.npz")
summary = pd.DataFrame([
    {"array_key": key, "shape": data[key].shape}
    for key in data.files
])
print(summary)